In [1]:
#!pip install librosa tensorflow scikit-learn

import os
import numpy as np
import librosa
import tensorflow as tf
from sklearn.model_selection import train_test_split

In [3]:
!git clone https://github.com/Jakobovski/free-spoken-digit-dataset.git

Cloning into 'free-spoken-digit-dataset'...
Updating files:  26% (806/3014)
Updating files:  27% (814/3014)
Updating files:  28% (844/3014)
Updating files:  29% (875/3014)
Updating files:  30% (905/3014)
Updating files:  31% (935/3014)
Updating files:  32% (965/3014)
Updating files:  33% (995/3014)
Updating files:  34% (1025/3014)
Updating files:  35% (1055/3014)
Updating files:  36% (1086/3014)
Updating files:  37% (1116/3014)
Updating files:  38% (1146/3014)
Updating files:  39% (1176/3014)
Updating files:  40% (1206/3014)
Updating files:  41% (1236/3014)
Updating files:  42% (1266/3014)
Updating files:  43% (1297/3014)
Updating files:  44% (1327/3014)
Updating files:  45% (1357/3014)
Updating files:  46% (1387/3014)
Updating files:  47% (1417/3014)
Updating files:  48% (1447/3014)
Updating files:  49% (1477/3014)
Updating files:  50% (1507/3014)
Updating files:  51% (1538/3014)
Updating files:  52% (1568/3014)
Updating files:  52% (1592/3014)
Updating files:  53% (1598/3014)
Updatin

In [4]:
DATA_PATH = "free-spoken-digit-dataset/recordings"

SAMPLE_RATE = 8000
MFCC_NUM = 13
MAX_LEN = 32

def extract_features(file_path):
    audio, sr = librosa.load(file_path, sr=SAMPLE_RATE)
    mfcc = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=MFCC_NUM)
    
    if mfcc.shape[1] < MAX_LEN:
        mfcc = np.pad(mfcc, ((0,0),(0,MAX_LEN - mfcc.shape[1])))
    else:
        mfcc = mfcc[:, :MAX_LEN]
    
    return mfcc

In [5]:
X = []
y = []
file_paths = []

for file in os.listdir(DATA_PATH):
    if file.endswith(".wav"):
        label = int(file.split("_")[0])
        path = os.path.join(DATA_PATH, file)
        
        X.append(extract_features(path))
        y.append(label)
        file_paths.append(path)

X = np.array(X)[..., np.newaxis]
y = np.array(y)

X_train, X_test, y_train, y_test, paths_train, paths_test = train_test_split(
    X, y, file_paths, test_size=0.2, random_state=42
)

C:\Users\senth\AppData\Roaming\Python\Python313\site-packages\librosa\core\spectrum.py:266: UserWarning: n_fft=2048 is too large for input signal of length=1933
  warnings.warn(
C:\Users\senth\AppData\Roaming\Python\Python313\site-packages\librosa\core\spectrum.py:266: UserWarning: n_fft=2048 is too large for input signal of length=1399
  warnings.warn(
C:\Users\senth\AppData\Roaming\Python\Python313\site-packages\librosa\core\spectrum.py:266: UserWarning: n_fft=2048 is too large for input signal of length=1876
  warnings.warn(
C:\Users\senth\AppData\Roaming\Python\Python313\site-packages\librosa\core\spectrum.py:266: UserWarning: n_fft=2048 is too large for input signal of length=1983
  warnings.warn(
C:\Users\senth\AppData\Roaming\Python\Python313\site-packages\librosa\core\spectrum.py:266: UserWarning: n_fft=2048 is too large for input signal of length=2033
  warnings.warn(
C:\Users\senth\AppData\Roaming\Python\Python313\site-packages\librosa\core\spectrum.py:266: UserWarning: n_fft

In [6]:
model = tf.keras.Sequential([
    tf.keras.layers.Conv2D(32, (3,3), activation='relu', input_shape=(13, 32, 1)),
    tf.keras.layers.MaxPooling2D((2,2)),
    
    tf.keras.layers.Conv2D(64, (3,3), activation='relu'),
    tf.keras.layers.MaxPooling2D((2,2)),
    
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dense(10, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.fit(X_train, y_train, epochs=15, batch_size=16, validation_split=0.1)

C:\Users\senth\AppData\Roaming\Python\Python313\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/15
135/135 ━━━━━━━━━━━━━━━━━━━━ 5s 11ms/step - accuracy: 0.6125 - loss: 1.2871 - val_accuracy: 0.8167 - val_loss: 0.4943
Epoch 2/15
135/135 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.8935 - loss: 0.3195 - val_accuracy: 0.9208 - val_loss: 0.2278
Epoch 3/15
135/135 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.9426 - loss: 0.1765 - val_accuracy: 0.9500 - val_loss: 0.1560
Epoch 4/15
135/135 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.9444 - loss: 0.1645 - val_accuracy: 0.8708 - val_loss: 0.3147
Epoch 5/15
135/135 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.9532 - loss: 0.1296 - val_accuracy: 0.9500 - val_loss: 0.1570
Epoch 6/15
135/135 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.9676 - loss: 0.0945 - val_accuracy: 0.9500 - val_loss: 0.1203
Epoch 7/15
135/135 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.9810 - loss: 0.0678 - val_accuracy: 0.9500 - val_loss: 0.1153
Epoch 8/15
135/135 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.9694 - loss: 0.0984 - val_accuracy: 0

In [7]:
loss, acc = model.evaluate(X_test, y_test)
print("Test Accuracy:", acc)

model.save("digit_asr_model.h5")

19/19 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.9617 - loss: 0.1355


Test Accuracy: 0.9616666436195374


In [8]:
model = tf.keras.models.load_model("digit_asr_model.h5")

In [10]:
import random
import os

def predict_digit(file_path):
    mfcc = extract_features(file_path)
    mfcc = mfcc[np.newaxis, ..., np.newaxis]
    
    pred = model.predict(mfcc, verbose=0)
    digit = np.argmax(pred)
    
    # Try to get actual label (only works for dataset files)
    try:
        actual = int(os.path.basename(file_path).split("_")[0])
        print("Actual Digit:", actual)
    except:
        print("Actual Digit: Unknown (custom audio)")
    
    print("Predicted Digit:", digit)
    print("File Used:", file_path)


# # 🔹 OPTION 1: Random dataset audio
# sample_path = random.choice(paths_test)
# print("\n--- Testing with DATASET sample ---")
# predict_digit(sample_path)


# 🔹 OPTION 2: Give your own audio file path
# (uncomment and provide path)

custom_path = r"PTT-20260329-WA0009.wav"
print("\n--- Testing with CUSTOM audio ---")
predict_digit(custom_path)


--- Testing with CUSTOM audio ---
Actual Digit: Unknown (custom audio)
Predicted Digit: 9
File Used: PTT-20260329-WA0009.wav
